# PRAGMA Phase 7 - Packed varlen execution

Implements `VarLenAttentionBackend` (section 9.2; ADR 0006, ADR 0010) using PyTorch's built-in nested-tensor (`torch.jagged` layout) support for `scaled_dot_product_attention` — the exact mechanism the implementation plan's own references point to (section 22, "PyTorch: Using variable-length attention"). No new third-party dependency: it works identically on CPU and CUDA, which matters here since this machine has no usable CUDA PyTorch build (`CLAUDE.md`'s environment note).

Checks the Phase 7 exit gate directly: **forward/backward parity with the padded backend**, **no attention crosses event or customer boundaries**, and **a meaningful memory improvement on representative (skewed) lengths** — see the methodology note below on why this notebook measures memory rather than wall-clock time.

In [1]:
from pathlib import Path

import torch

from pragma.attention import PaddedAttentionBackend, VarLenAttentionBackend, compare_attention_cost
from pragma.config import MaskingConfig
from pragma.data import ParquetShardStore, PragmaCollator, TokenizedRecordDataset
from pragma.masking import MaskingPlanner
from pragma.modeling import PragmaConfig, PragmaForMaskedModeling
from pragma.processing import PragmaProcessor
from pragma.schema import SchemaRegistry

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

registry = SchemaRegistry.default()
processor = PragmaProcessor.load(REPO_ROOT / "data" / "processor", registry)
train_dataset = TokenizedRecordDataset.from_store(
    REPO_ROOT / "data" / "shards", ParquetShardStore(), split="train"
)
print(f"{len(train_dataset)} train records")

399 train records


## Build a real masked batch and a small model

In [2]:
planner = MaskingPlanner.from_registry(registry, processor.key_vocab, MaskingConfig())
collator = PragmaCollator(masking_planner=planner)
records = [train_dataset[i] for i in range(24) if len(train_dataset[i].events) > 0]
batch = collator(records)

config = PragmaConfig.from_processor(
    processor, hidden_size=24, num_heads=2, intermediate_size=48, event_layers=2, history_layers=1
)
torch.manual_seed(0)
model = PragmaForMaskedModeling(config)
model.eval()
print(f"n_records={batch.n_records}, n_events={batch.n_events}, n_event_tokens={batch.n_event_tokens}")

n_records=10, n_events=283, n_event_tokens=1501


## Exit gate: forward parity

Same model, same batch, only the attention backend differs.

In [3]:
with torch.no_grad():
    out_padded = model(batch, attention_backend=PaddedAttentionBackend())
    out_varlen = model(batch, attention_backend=VarLenAttentionBackend())

record_diff = (out_padded.record_embeddings - out_varlen.record_embeddings).abs().max().item()
event_diff = (out_padded.event_embeddings - out_varlen.event_embeddings).abs().max().item()
logits_diff = (out_padded.mlm_logits - out_varlen.mlm_logits).abs().max().item()
print(f"max record_embeddings diff: {record_diff:.2e}")
print(f"max event_embeddings diff:  {event_diff:.2e}")
print(f"max mlm_logits diff:        {logits_diff:.2e}")
print(f"loss padded={out_padded.loss.item():.4f}  loss varlen={out_varlen.loss.item():.4f}")

assert torch.allclose(out_padded.record_embeddings, out_varlen.record_embeddings, atol=1e-5)
assert torch.allclose(out_padded.mlm_logits, out_varlen.mlm_logits, atol=1e-4)

max record_embeddings diff: 1.19e-07
max event_embeddings diff:  1.19e-07
max mlm_logits diff:        5.59e-09
loss padded=5.9540  loss varlen=5.9540


C:\Users\levyr\Desktop\random-projects\pragma\.venv\Lib\site-packages\torch\nested\__init__.py:119: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\aten\src\ATen\NestedTensorImpl.cpp:181.)
  return torch._nested_tensor_from_tensor_list(ts, dtype, None, device, None)


## Exit gate: backward parity

In [4]:
import copy

model_padded = model
model_varlen = copy.deepcopy(model)

model_padded(batch, attention_backend=PaddedAttentionBackend()).loss.backward()
model_varlen(batch, attention_backend=VarLenAttentionBackend()).loss.backward()

max_grad_diff = max(
    (p1.grad - p2.grad).abs().max().item()
    for p1, p2 in zip(model_padded.parameters(), model_varlen.parameters(), strict=True)
)
print(f"max gradient diff across all parameters: {max_grad_diff:.2e}")
assert max_grad_diff < 1e-2

max gradient diff across all parameters: 9.31e-10


## Exit gate: no attention crosses event/customer boundaries

Same isolation check as `006_model_architecture.ipynb`, now with `VarLenAttentionBackend`: an event's contextual embedding must not depend on which unrelated events/records happen to share its packed batch.

In [5]:
target = next(r for r in train_dataset._records if len(r.events) >= 2)
others = [
    r for r in train_dataset._records if r.entity_id != target.entity_id and r.events
][:6]

plain_collator = PragmaCollator()
solo_batch = plain_collator([target])
group_batch = plain_collator(others + [target])

with torch.no_grad():
    _, solo_event_emb, _ = model.pragma(solo_batch, attention_backend=VarLenAttentionBackend())
    _, group_event_emb, _ = model.pragma(group_batch, attention_backend=VarLenAttentionBackend())

n_target_events = len(target.events)
group_target_events = group_event_emb[-n_target_events:]
max_diff = (solo_event_emb - group_target_events).abs().max().item()
print(f"max abs difference (solo vs. inside a larger batch): {max_diff:.2e}")
assert torch.allclose(solo_event_emb, group_target_events, atol=1e-5)

max abs difference (solo vs. inside a larger batch): 5.96e-08


## Exit gate: memory improvement on representative (skewed) lengths

**Methodology note:** the direct approach — time both backends on a skewed batch and check varlen is faster — does not hold on this CPU-only dev machine: PyTorch's nested-tensor (`torch.jagged`) SDPA kernel is not yet as optimized on CPU as the dense batched math path `PaddedAttentionBackend` uses, so wall-clock time can actually *regress* under `VarLenAttentionBackend` here even though it computes over far fewer elements (confirmed directly below, not assumed — this is the honest finding, not a hidden problem). The *element-count* comparison (`pragma.attention.compare_attention_cost`) is what actually backs the exit gate: it is deterministic, hardware-independent, and is exactly where the real memory/compute saving comes from — GPU kernels (FlashAttention et al.) are expected to translate this saving into wall-clock speedup once a compatible GPU is available (ADR 0010).

In [6]:
import random

rng = random.Random(0)
short_events = [rng.randint(1, 6) for _ in range(200)]
long_events = [24] * 10  # config.max_event_tokens
lengths = short_events + long_events

comparison = compare_attention_cost(lengths, num_heads=3, head_dim=64)
print(comparison)
print(f"qkv memory ratio (padded / packed): {comparison.qkv_memory_ratio:.1f}x")
print(f"attention-score memory ratio:        {comparison.score_memory_ratio:.1f}x")
assert comparison.qkv_memory_ratio > 2.0
assert comparison.score_memory_ratio > 5.0

AttentionCostComparison(n_sequences=210, total_tokens=924, max_length=24, padded_qkv_elements=967680, packed_qkv_elements=177408, padded_score_elements=362880, packed_score_elements=26160)
qkv memory ratio (padded / packed): 5.5x
attention-score memory ratio:        13.9x


### Honest wall-clock finding on this CPU-only machine

In [7]:
import time

n = sum(lengths)
cu = torch.tensor([0, *torch.tensor(lengths).cumsum(0).tolist()])
q = torch.randn(n, 3, 64)
k = torch.randn(n, 3, 64)
v = torch.randn(n, 3, 64)

padded_backend = PaddedAttentionBackend()
varlen_backend = VarLenAttentionBackend()
padded_backend.forward(q, k, v, cu)  # warmup
varlen_backend.forward(q, k, v, cu)

n_iters = 5
t0 = time.perf_counter()
for _ in range(n_iters):
    padded_backend.forward(q, k, v, cu)
t_padded = (time.perf_counter() - t0) / n_iters

t0 = time.perf_counter()
for _ in range(n_iters):
    varlen_backend.forward(q, k, v, cu)
t_varlen = (time.perf_counter() - t0) / n_iters

print(f"padded: {t_padded * 1000:.2f} ms/iter")
print(f"varlen: {t_varlen * 1000:.2f} ms/iter")
print(
    f"On this CPU, varlen is {'faster' if t_varlen < t_padded else 'slower'} "
    f"despite using {comparison.score_memory_ratio:.0f}x fewer attention-score elements "
    "-- expected on CPU per the methodology note above, revisit once a CUDA GPU is available."
)

padded: 8.04 ms/iter
varlen: 39.90 ms/iter
On this CPU, varlen is slower despite using 14x fewer attention-score elements -- expected on CPU per the methodology note above, revisit once a CUDA GPU is available.
